<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Meet Your Duckiebot

In this notebook you will learn about the three hardware components of the Duckiebot that you will control in this exercise:

1. **Wheels** — differential drive motors
2. **Camera** — forward-facing RGB camera
3. **LEDs** — four RGB corner lights


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Wheels — Differential Drive

Your Duckiebot steers by spinning each wheel at a different speed. This is called **differential drive**.

The API call is:
```python
wheels.set_wheels_speed(left, right)
```

Speeds are in the range `[-1.0, 1.0]` — positive means forward, negative means backward.

| Left | Right | Result |
|------|-------|--------|
| +0.5 | +0.5 | Straight forward |
| -0.5 | -0.5 | Straight backward |
| +0.5 | -0.5 | Spin right |
| -0.5 | +0.5 | Spin left |
| +0.5 | +0.2 | Curve right |


The key insight: the robot turns **toward the slower wheel**.

To see this intuitively, think of the robot as pivoting around a point — if one wheel stops completely, the robot pivots around that wheel.


In [ ]:

examples = [
    (0.5,  0.5,  "Forward"),
    (-0.5, -0.5, "Backward"),
    (0.5,  -0.5, "Spin right"),
    (0.5,  0.2,  "Curve right"),
]

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, (left, right, label) in zip(axes, examples):
    ax.bar(['Left', 'Right'], [left, right], color=['blue', 'orange'])
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(label)
    ax.set_ylabel('Speed')
plt.tight_layout()
plt.show()

## 2. Camera

The Duckiebot has a forward-facing camera that captures images at **640×480 pixels** in **RGB format**.

The API call is:
```python
success, frame = camera.read_rgb()
```

`frame` is a NumPy array with shape `(height, width, 3)` and values 0–255.


In [ ]:

fn = '../../../assets/samples/big-duck/big-duck-08.jpg'
frame = plt.imread(fn)
print('shape:', frame.shape)   # (height, width, channels)
print('dtype:', frame.dtype)
plt.imshow(frame)
plt.axis('off')
plt.title('Sample camera frame')
plt.show()

In [ ]:
# Each pixel has three channels: Red, Green, Blue
R = frame[:, :, 0]
G = frame[:, :, 1]
B = frame[:, :, 2]

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(R, cmap='Reds')
axes[0].set_title('Red channel')
axes[1].imshow(G, cmap='Greens')
axes[1].set_title('Green channel')
axes[2].imshow(B, cmap='Blues')
axes[2].set_title('Blue channel')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. LEDs

Your Duckiebot has **4 RGB LEDs** at the corners:

```
      FRONT
 [0] ─────── [2]
  |           |
 [4] ─────── [3]
      BACK
```

The API call is:
```python
leds.set_rgb(led_index, [r, g, b])
```

Colors are in the range `[0.0, 1.0]` — for example:
- `[1.0, 0.0, 0.0]` = red
- `[1.0, 1.0, 0.0]` = yellow
- `[1.0, 1.0, 1.0]` = white
- `[0.0, 0.0, 0.0]` = off

The functions you will implement return a **dictionary** mapping LED index to color:
```python
{0: [1.0, 1.0, 0.0],   # front-left = yellow
 2: [0.0, 0.0, 0.0],   # front-right = off
 3: [1.0, 1.0, 0.0],   # back-right = yellow
 4: [0.0, 0.0, 0.0]}   # back-left = off
```




In [ ]:
# Visualize: a 2x2 grid of colored squares representing the 4 LEDs
# Positions: top-left=LED0, top-right=LED2, bottom-left=LED3, bottom-right=LED4
left_turn = {
    0: [1.0, 1.0, 0.0],  # front-left yellow
    2: [0.0, 0.0, 0.0],  # front-right off
    3: [1.0, 1.0, 0.0],  # back-right yellow
    4: [0.0, 0.0, 0.0],  # back-left off
}

fig, ax = plt.subplots(figsize=(3, 3))
positions = {0: (0, 1), 2: (1, 1), 3: (0, 0), 4: (1, 0)}
labels    = {0: 'LED 0\nfront-left', 2: 'LED 2\nfront-right',
             3: 'LED 3\nback-right',  4: 'LED 4\nback-left'}

for idx, (col, row) in positions.items():
    color = left_turn[idx]
    ax.add_patch(plt.Rectangle((col, row), 0.9, 0.9, color=color,
                               edgecolor='black', linewidth=1))
    ax.text(col + 0.45, row + 0.45, labels[idx], ha='center', va='center',
            fontsize=7, color='black')

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_aspect('equal')
ax.set_title('Left turn signal (yellow = on)')
ax.axis('off')
plt.show()

# Activity

In the `packages/led_control.py` file, implement the `set_turning_leds()`. String will be given from one of those values [ 'left', 'right', 'forward' , 'stop' ]